# 🔄 Grundlegende Agenten-Workflows mit GitHub-Modellen (Python)

## 📋 Tutorial zur Workflow-Orchestrierung

Dieses Notebook stellt die leistungsstarken **Workflow Builder**-Funktionen des Microsoft Agent Frameworks vor. Lernen Sie, wie Sie ausgeklügelte, mehrstufige Agenten-Workflows erstellen, die komplexe Geschäftsprozesse bewältigen und mehrere KI-Operationen nahtlos koordinieren können.

## 🎯 Lernziele

### 🏗️ **Workflow-Architektur**
- **Workflow Builder**: Entwerfen und orchestrieren komplexer mehrstufiger Prozesse
- **Ereignisgesteuerte Ausführung**: Umgang mit Workflow-Ereignissen und Zustandsübergängen
- **Visuelles Workflow-Design**: Erstellen und Visualisieren von Workflow-Strukturen
- **GitHub-Modelle-Integration**: Nutzung von KI-Modellen im Workflow-Kontext

### 🔄 **Prozessorchestrierung**
- **Sequentielle Operationen**: Verkettung mehrerer Agentenaufgaben in logischer Reihenfolge
- **Bedingte Logik**: Umsetzung von Entscheidungspunkten und verzweigten Workflows
- **Fehlerbehandlung**: Robuste Fehlerwiederherstellung und Workflow-Resilienz
- **Zustandsverwaltung**: Nachverfolgung und Verwaltung des Workflow-Ausführungszustands

### 📊 **Enterprise-Workflow-Muster**
- **Automatisierung von Geschäftsprozessen**: Automatisierung komplexer organisatorischer Workflows
- **Koordination mehrerer Agenten**: Koordination mehrerer spezialisierter Agenten
- **Skalierbare Ausführung**: Design von Workflows für unternehmensweite Operationen
- **Monitoring & Beobachtbarkeit**: Überwachung der Workflow-Leistung und Ergebnisse

## ⚙️ Voraussetzungen & Einrichtung

### 📦 **Benötigte Abhängigkeiten**

Installieren Sie das Agent Framework mit Workflow-Fähigkeiten:

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub-Modelle-Konfiguration**

**Umgebungseinrichtung (.env Datei):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

### 🏢 **Enterprise-Anwendungsfälle**

**Beispiele für Geschäftsprozesse:**
- **Kunden-Onboarding**: mehrstufige Verifizierungs- und Einrichtung-Workflows
- **Content-Pipeline**: Automatisierte Erstellung, Überprüfung und Veröffentlichung von Inhalten
- **Datenverarbeitung**: ETL-Workflows mit KI-gestützter Transformation
- **Qualitätssicherung**: Automatisierte Test- und Validierungsprozesse

**Workflow-Vorteile:**
- 🎯 **Zuverlässigkeit**: Deterministische Ausführung mit Fehlerwiederherstellung
- 📈 **Skalierbarkeit**: Bewältigung der Prozessautomatisierung in großem Volumen
- 🔍 **Beobachtbarkeit**: Vollständige Prüfpfade und Monitoring
- 🔧 **Wartbarkeit**: Visuelles Design und modulare Komponenten

## 🎨 Workflow-Designmuster

### Grundstruktur eines Workflows
```mermaid
graph TD
    A[Start] --> B[Aufgabe des Agenten 1]
    B --> C{Entscheidungspunkt}
    C -->|Erfolg| D[Aufgabe des Agenten 2]
    C -->|Fehler| E[Fehlerbehandlung]
    D --> F[Ende]
    E --> F
```

**Schlüsselkomponenten:**
- **WorkflowBuilder**: Haupt-Orchestrierungs-Engine
- **WorkflowEvent**: Ereignisbehandlung und Kommunikation
- **WorkflowViz**: Visuelle Workflow-Darstellung und Debugging

Lassen Sie uns Ihren ersten intelligenten Workflow erstellen! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load GitHub Models API credentials from .env file
load_dotenv()

In [ ]:
# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = await provider.create_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = await provider.create_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
